In [20]:
!pip install -q langchain langchain-classic langchain-community langchain-huggingface pypdf faiss-cpu sentence-transformers transformers torch accelerate

In [21]:
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


print("Please click 'Choose Files' to upload your U.S. Financial Report PDF:")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f"\n Successfully uploaded: {pdf_filename}")


loader = PyPDFLoader(pdf_filename)
documents = loader.load()
print(f"Loaded {len(documents)} page(s) from {pdf_filename}.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks.")

Please click 'Choose Files' to upload your U.S. Financial Report PDF:


Saving US_Financial_Report.pdf to US_Financial_Report (2).pdf

 Successfully uploaded: US_Financial_Report (2).pdf
Loaded 160 page(s) from US_Financial_Report (2).pdf.
Created 1372 text chunks.


In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embedding_model)
print(f"Vector Database built successfully with {vector_store.index.ntotal} vectors.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Database built successfully with 1372 vectors.


In [23]:
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


model_id = "Qwen/Qwen2.5-1.5B-Instruct"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    max_new_tokens=250,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=pipe)


retriever = vector_store.as_retriever(search_kwargs={"k": 3})


system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you do not know the answer, say that you "
    "don't know. Keep your answer concise.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print(" RAG Pipeline ready for queries")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

 RAG Pipeline ready for queries


In [24]:
# Test Query on U.S. Financial Report
query = "What are the key financial highlights or net costs discussed in this report?"
response = rag_chain.invoke({"input": query})

print("--- Question ---")
print(query)

print("\n--- Answer ---")
print(response["answer"])

print("\n--- Retrieved Source Contexts ---")
for i, doc in enumerate(response["context"]):
    print(f"\n[Chunk {i+1} - Page {doc.metadata.get('page', 'N/A')}]")
    print(doc.page_content)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Question ---
What are the key financial highlights or net costs discussed in this report?

--- Answer ---
System: You are a helpful assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you do not know the answer, say that you don't know. Keep your answer concise.

Context:
been particularly challenging to 
implement.175 
Many of the benefits and costs 
discussed below are difficult to 
quantify. In some cases, data needed to 
quantify these economic effects are not 
currently available and the SEC does not 
have information or data that would 
allow such quantification. For example, 
while we anticipate that the quantified 
cost-savings estimates would apply 
broadly for each category of private fund 
adviser, these estimates depend on

our selection of data sources, empirical 
methodology, and the assumptions the 
SEC has made throughout the analysis. 
Commenters are requested to provide 
empirical data, estimation 
metho